### Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import mne
import os
import time



### Set up data path and read data

In [ ]:
DATA_PATH = "../data/raw/workload_dataset/data_n_back_test/eeg/eeg.parquet"
TIMESTAMP_COL = "timestamp" # found from /explore_raw_data notebook

df = pd.read_parquet(DATA_PATH)
eeg_columns = [col for col in df.columns if col.startswith("EEG")]
# removing eeg columns that aren't relevant 
eeg_columns = [c for c in eeg_columns if c not in [
        "EEG.Counter",
        "EEG.Interpolated",
        "EEG.RawCq",
        "EEG.Battery",
        "EEG.MarkerHardware",
    ]
]

### Sampling rate check

In [ ]:
SFREQ = 128.0 # Assuming a fixed sampling frequuency of 128 Hz for the Emotiv EPOC device.
# Found from www.mdpi.com/1424-8820/24/4/1174 

def check_plausible_duration(segment, sfreq=SFREQ, min_dur=5, max_dur=3600):
      """
      Check if the duration of the segment is within plausible limits.
      
      Parameters:
         segment: DataFrame containing the EEG data for a specific subject, test, and phase.
         sfreq: Sampling frequency in Hz.
         min_dur: Minimum plausible duration in seconds.
         max_dur: Maximum plausible duration in seconds.
      
      Returns:
         True if the duration is plausible, False otherwise.
      """
      
      duration = len(segment) / sfreq
      
      if duration < min_dur or duration > max_dur:
         return False, duration
      return True, duration

### Running through one example segment 


In [ ]:
subject, test, phase = "subject_05", 2, 2 
segment = df[(df["subject"] == subject) & (df["test"] == test) & (df["phase"] == phase)]
print("Segment shape is: ", segment.shape) # Double checking the shape of the segment

plausible, duration = check_plausible_duration(segment)
print(f"Duration of segment at {SFREQ} Hz is {duration:.2f}s ({duration/60:.2f} min) and plausible: {plausible}")



In [ ]:
data_v = segment[eeg_columns].to_numpy().T * 1e-6 # Convert from microvolts to volts 
# Converted to volts so it's easier to look at the data

info = mne.create_info(ch_names=eeg_columns, sfreq=SFREQ, ch_types="eeg")
raw = mne.io.RawArray(data_v, info, verbose=False)

# Chosen 0.1 and 40.0 Hz as the filters based off typical cutoff filters
# From https://www.pressrelease.brainproducts.com/eeg-artificats-handling-in-analyzer 
raw.filter(l_freq=0.1, h_freq=40.0, verbose=False)
# No notch filter needed because data was already parsed for 50/60 Hz
epochs = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=0.0, preload=True, verbose=False)
print(epochs) # Checking to see if the epochs were created correctly 

### Finding the rejection threshold


In [ ]:
data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times) unit: volts
ptp = data.max(axis=2) - data.min(axis=2)  # peak-to-peak amplitude for each epoch and channel

# Temporarily includes original microvolt units
print("Pooling across all channels: ")
for p in [25, 50, 75, 90, 83, 95, 99]:
     print(f"{p}th percentile: {np.percentile(ptp, p) * 1e6:.2f} uV")

print("Median peak-to-peak amplitude for each channel: ")
for ch_name, ch_ptp in zip(eeg_columns, ptp.T):
     print(f" {ch_name}: {np.median(ch_ptp) * 1e6:.2f} uV")

In [ ]:
# Rejecting the epochs that are outside of the 95th percentile of peak-to-peak amplitude
REJECT_THRESHOLD = np.percentile(ptp, 95) 
print(f"Using rejection threshold: {REJECT_THRESHOLD * 1e6:.2f} uV")

epochs.drop_bad(reject=dict(eeg=REJECT_THRESHOLD))
print(f"Remaining epochs: {len(epochs)}")
print(f"Original number of epochs was: {len(ptp)}")


### Wrap everything into a function 

In [ ]:
def process_segment(df, subject, test, phase, channel_cols, reject_threshold, sfreq=SFREQ):
     """
     Process a segment of EEG data for a specific subject, test, and phase.
     
     Parameters:
         df: DataFrame containing the EEG data.
         subject: Subject identifier.
         test: Test identifier.
         phase: Phase identifier.
         channel_cols: List of EEG channel column names.
         reject_threshold: Peak-to-peak amplitude threshold for rejecting epochs (in volts).
         sfreq: Sampling frequency in Hz.
     
     Returns:
          epochs: MNE Epochs object containing the processed EEG data.
     """
     
     segment = df[(df["subject"] == subject) & (df["test"] == test) & (df["phase"] == phase)]
     # Decided that 5 rows is the min number of rows to process a segment
     if len(segment) < 5:
          return None, "too few rows"
     plausible, duration = check_plausible_duration(segment, sfreq)
     if not plausible:
          return None, f"duration {duration:.2f}s not plausible"
     
     # Convert to volts for easier processing
     data_v = segment[channel_cols].to_numpy().T * 1e-6 # Convert from microvolts to volts
     info = mne.create_info(ch_names=channel_cols, sfreq=sfreq, ch_types="eeg", verbose=False)
     raw = mne.io.RawArray(data_v, info, verbose=False)
     raw.filter(l_freq=1.0, h_freq=40.0, verbose=False)
     
     # Making epochs and filtering them based on the rejection threshold
     epochs = mne.make_fixed_length_epochs(raw, duration=2.0, overlap=0.0, preload=True, verbose=False)
     epochs.drop_bad(reject=dict(eeg=reject_threshold))
     
     if len(epochs) == 0:
          return None, "all epochs rejected"
     return epochs, len(epochs)
     

### Run through all 144 combinations in data

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

skip_reasons = {}
saved_count = 0
start_time = time.time()

subjects = sorted(df["subject"].unique())
tests = sorted(df["test"].unique())
phases = sorted(df["phase"].unique())
total = len(subjects) * len(tests) * len(phases)
done = 0

for sub in subjects:
     for test in tests:
          for phase in phases:
               done += 1
               result, status = process_segment(df, sub, test, phase, eeg_columns, REJECT_THRESHOLD)
               if result is not None:
                    filename = os.path.join("../data/processed", f"{sub}_test{test}_phase{phase}-epo.fif")
                    result.save(filename, overwrite=True, verbose=False)
                    saved_count += 1
                    print(f"({done}/{total}) Saved {filename} with {len(result)} epochs")
               else:
                    skip_reasons[status] = skip_reasons.get(status, 0) + 1
                    print(f"({done}/{total}) Skipped subject {sub}, test {test}, phase {phase}: {status}")

elapsed = time.time() - start_time
print(f"\nPreprocessing complete. Time elapsed: {elapsed:.2f}s\nSaved {saved_count}/{total}")
print(f"Skip reasons: {skip_reasons}")


## Final notes

I chose the rejection threshold of 95% because I wanted to include the AF3 and AF4 which were the highest channels. Those two channels could be considered outliers in another context but I felt that the AF3 and AF4 were necessary to include because this analysis regards a stressful activity but I didn't want to include everything. A threshold of anywhere between 90 and 95% would result in good data according to my analysis.

143 epochs were saved and 1 was rejected because it had too little rows.
The 1 rejected epoch was subject 9, test 1, phase 3.

